In [ ]:
if not law_preds_test:
    print('Skipping submission generation: law_predictions_per_query_TEST.json not found.')
    print('Run the law pipeline on test.csv first, then re-execute this cell.')
else:
    # Reuse run_judge logic on test by temporarily pointing law_preds_val and val_query at test
    JUDGE_TEST_CACHE = OUT_DIR / 'judge_yes_scores_TEST.json'
    test_cache = {}
    if JUDGE_TEST_CACHE.exists():
        test_cache = json.loads(JUDGE_TEST_CACHE.read_text(encoding='utf-8'))

    sp = SamplingParams(max_tokens=1, temperature=0.0, logprobs=20)
    test_scores = {}
    t0 = time.time()
    for qid in sorted(test_query):
        cands = law_preds_test.get(qid, [])
        if not cands:
            test_scores[qid] = {}; continue
        if qid in test_cache and len(test_cache[qid]) == len(cands):
            test_scores[qid] = test_cache[qid]; continue
        prompts = []
        for c in cands:
            txt, ti, resolved = lookup_law_text(c, max_chars=900)
            if not txt: txt = '(no statute text available — judge must reason from the citation string only)'
            user = USER_TEMPLATE.format(query=test_query[qid][:2000], citation=c, law_text=txt, law_title=ti or '(no title)')
            prompts.append(TOK.apply_chat_template(
                [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user}],
                tokenize=False, add_generation_prompt=True))
        outputs = LLM.generate(prompts, sp)
        scores = {}
        for c, o in zip(cands, outputs):
            lp_dict = {}
            if o.outputs and o.outputs[0].logprobs:
                for tok_id, lp_obj in o.outputs[0].logprobs[0].items():
                    if hasattr(lp_obj, 'decoded_token'):
                        lp_dict[lp_obj.decoded_token] = lp_obj.logprob
                    else:
                        lp_dict[str(tok_id)] = float(lp_obj)
            scores[c] = softmax_yes(lp_dict)
        test_scores[qid] = scores
        test_cache[qid] = scores
        JUDGE_TEST_CACHE.write_text(json.dumps(test_cache, ensure_ascii=False, indent=1), encoding='utf-8')
        print(f'  {qid}: {len(scores)} candidates, elapsed={time.time()-t0:.1f}s')

    # Apply best tau, canonicalize for submission
    rows = []
    for qid in sorted(test_query):
        cands_kept = [c for c, s in test_scores.get(qid, {}).items() if s >= best_tau]
        # canonicalize (the official grader does this anyway, but it produces a tidier file)
        cands_canon = canon_list(cands_kept)
        rows.append({'query_id': qid, 'predicted_citations': ';'.join(cands_canon)})

    sub_df = pd.DataFrame(rows)
    SUB_PATH = OUT_DIR / 'submission_judge_law_only.csv'
    sub_df.to_csv(SUB_PATH, index=False)
    print(f'\nWrote {SUB_PATH}  ({len(sub_df)} rows)')
    print(sub_df.head(3).to_string(index=False))


## Cell 10 — Generate test submission with the best τ

When ready, run the law pipeline on `test.csv` to save `law_predictions_per_query_TEST.json`, then re-run this cell. It applies the same judge + threshold to test candidates and writes `submission.csv` in the official format.


In [ ]:
TAU = best_tau

for qid in sorted(val_gold):
    scores = all_scores.get(qid, {})
    gold_canon = set(canon_list(val_gold[qid]))
    cand_canon_pairs = [(c, NORMALIZER.canonicalize(c), s) for c, s in scores.items()]
    accepted = [(c, can, s) for c, can, s in cand_canon_pairs if s >= TAU and can]
    rejected = [(c, can, s) for c, can, s in cand_canon_pairs if s < TAU and can]

    tp_set = {can for _, can, _ in accepted if can in gold_canon}
    fp_set = {can for _, can, _ in accepted if can not in gold_canon}
    fn_set = gold_canon - {can for _, can, _ in accepted}

    print(f'\n### {qid}  |gold_canon|={len(gold_canon)}  |accepted|={len({can for _, can, _ in accepted})}  TP={len(tp_set)}  FP={len(fp_set)}  FN={len(fn_set)}')

    if accepted:
        print('  accepted (sorted by yes-score):')
        for c, can, s in sorted(accepted, key=lambda x: -x[2])[:12]:
            tag = 'TP' if can in gold_canon else 'FP'
            print(f'   [{tag}] {s:.3f}  {c}  -> {can}')

    if rejected:
        # show high-confidence rejections that were actually gold (missed-by-judge errors)
        missed = [(c, can, s) for c, can, s in rejected if can in gold_canon]
        if missed:
            print('  rejected-but-gold (judge errors, low yes-score on TPs):')
            for c, can, s in sorted(missed, key=lambda x: -x[2])[:6]:
                print(f'   [MISS] {s:.3f}  {c}  -> {can}')


## Cell 9 — Per-query diagnostic: what did the judge actually do?

For the best τ: list each query's TP / FP / FN, sorted by yes-score. Helps debug whether the judge is removing the right FPs.


In [ ]:
# Run judge on all 10 val queries
all_scores = run_judge()  # uses cache if available
total_calls = sum(len(v) for v in all_scores.values())
print(f'\nJudge complete: {total_calls} verdicts across {len(all_scores)} queries')

# Sweep threshold
print('\n--- threshold sweep (official canonical macro F1) ---')
print(f"{'tau':>6} {'mean |P|':>10} {'macro_P':>10} {'macro_R':>10} {'macro_F1':>10}")
results = []
for tau in [0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    filtered = {
        q: [c for c, s in all_scores.get(q, {}).items() if s >= tau]
        for q in val_gold
    }
    macro, _ = official_score(filtered, val_gold, label=f'tau={tau}', show_per_query=False)
    mean_n = np.mean([len(filtered[q]) for q in val_gold])
    results.append((tau, mean_n, macro['macro_precision'], macro['macro_recall'], macro['macro_f1']))
    print(f"{tau:>6.2f} {mean_n:>10.1f} {macro['macro_precision']:>10.4f} {macro['macro_recall']:>10.4f} {macro['macro_f1']:>10.4f}")

best_tau, *_, best_f1 = max(results, key=lambda r: r[-1])
print(f'\nBest tau = {best_tau} -> macro F1 = {best_f1:.4f}')
print(f'Baseline (no judge): macro F1 = {macro_baseline["macro_f1"]:.4f}')
print(f'Lift from judge:    {best_f1 - macro_baseline["macro_f1"]:+.4f}')


## Cell 8 — Run judge on all val, then sweep threshold for best macro F1

Run the judge on every val query and every candidate. Then sweep the YES-score threshold τ over [0.30, 0.95] and pick the τ that maximises **official canonical macro F1**.


In [ ]:
SYSTEM_PROMPT = (
    "You are a Swiss legal expert. Your job is to decide whether a specific statute "
    "provision belongs in the citation list a Swiss Federal Court would write when "
    "answering a particular legal query. Be strict: include a provision only if its "
    "text directly addresses an issue raised by the facts in the query, OR if it is "
    "a procedural anchor that every appeal to the Federal Court must invoke. "
    "Reject provisions whose text is in the right general area but addresses a different issue. "
    "Answer with exactly one token: YES or NO."
)

USER_TEMPLATE = (
    "QUERY (English description of a Swiss legal scenario):\n{query}\n\n"
    "CANDIDATE CITATION:\n{citation}\n\n"
    "STATUTE TEXT (German):\n{law_text}\n\n"
    "STATUTE TITLE:\n{law_title}\n\n"
    "Question: Would the Federal Court drafting the opinion for this query cite this provision in its decision?\n"
    "Answer YES or NO."
)

def build_prompts(qid, candidates, max_law_chars=900, max_query_chars=2000):
    """Return list of dicts: {'cit', 'prompt', 'resolved'}."""
    q = val_query[qid] if qid in val_query else test_query.get(qid, '')
    q = q[:max_query_chars]
    out = []
    for c in candidates:
        txt, ti, resolved = lookup_law_text(c, max_chars=max_law_chars)
        if not txt:
            txt = '(no statute text available — judge must reason from the citation string only)'
        user = USER_TEMPLATE.format(query=q, citation=c, law_text=txt, law_title=ti or '(no title)')
        if TOK is not None:
            prompt = TOK.apply_chat_template(
                [{'role': 'system', 'content': SYSTEM_PROMPT},
                 {'role': 'user',   'content': user}],
                tokenize=False, add_generation_prompt=True
            )
        else:
            prompt = f'SYSTEM: {SYSTEM_PROMPT}\nUSER: {user}\nASSISTANT:'
        out.append({'cit': c, 'prompt': prompt, 'resolved': resolved})
    return out

# Cache for soft-yes scores
JUDGE_CACHE_PATH = OUT_DIR / 'judge_yes_scores.json'
judge_cache = {}
if JUDGE_CACHE_PATH.exists():
    judge_cache = json.loads(JUDGE_CACHE_PATH.read_text(encoding='utf-8'))
    print(f'Loaded judge cache: {sum(len(v) for v in judge_cache.values())} verdicts across {len(judge_cache)} queries')

import math

def softmax_yes(logprobs_token0):
    """Given a dict {token_str: logprob} from vLLM logprobs[0], return P(YES).
    Looks for both upper/lower-case YES/NO tokens (Qwen tokenizes them with leading space sometimes)."""
    yes_lp, no_lp = -1e9, -1e9
    for tok, lp in logprobs_token0.items():
        ts = tok.strip().lower()
        if ts in {'yes','y'}: yes_lp = max(yes_lp, lp)
        elif ts in {'no','n'}: no_lp = max(no_lp, lp)
    if yes_lp == -1e9 and no_lp == -1e9: return 0.5
    m = max(yes_lp, no_lp)
    e_yes, e_no = math.exp(yes_lp - m), math.exp(no_lp - m)
    return e_yes / (e_yes + e_no)

def run_judge(qid_subset=None, force=False, batch_size=64):
    """Judge every law candidate for every val query. Returns {qid: {cit: yes_score}}."""
    if LLM is None:
        print('LLM not loaded — cannot run judge. Returning empty.')
        return {q: {c: 0.5 for c in law_preds_val.get(q, [])} for q in val_gold}
    qids = list(val_gold) if qid_subset is None else qid_subset
    sp = SamplingParams(max_tokens=1, temperature=0.0, logprobs=20)
    out = {}
    t0 = time.time()
    for qid in qids:
        cands = law_preds_val.get(qid, [])
        if not cands:
            out[qid] = {}; continue
        if not force and qid in judge_cache and len(judge_cache[qid]) == len(cands):
            out[qid] = judge_cache[qid]
            continue
        prompts = build_prompts(qid, cands)
        scores = {}
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            outputs = LLM.generate([b['prompt'] for b in batch], sp)
            for b, o in zip(batch, outputs):
                lp_dict = {}
                if o.outputs and o.outputs[0].logprobs:
                    for tok_id, lp_obj in o.outputs[0].logprobs[0].items():
                        # vLLM returns Logprob objects; normalize to a plain dict
                        if hasattr(lp_obj, 'decoded_token'):
                            lp_dict[lp_obj.decoded_token] = lp_obj.logprob
                        else:
                            lp_dict[str(tok_id)] = float(lp_obj)
                scores[b['cit']] = softmax_yes(lp_dict)
        out[qid] = scores
        judge_cache[qid] = scores
        # persist incrementally
        JUDGE_CACHE_PATH.write_text(json.dumps(judge_cache, ensure_ascii=False, indent=1), encoding='utf-8')
        print(f"  {qid}: judged {len(scores)} candidates, mean yes-score = {np.mean(list(scores.values())):.3f}, elapsed={time.time()-t0:.1f}s")
    return out

# Smoke test on the first val query (judge only — does not score yet)
smoke_qid = sorted(val_gold)[0]
print(f'Smoke testing judge on {smoke_qid} ({len(law_preds_val.get(smoke_qid, []))} candidates)')
smoke_out = run_judge([smoke_qid])
for c, s in list(smoke_out[smoke_qid].items())[:10]:
    in_gold = c in set(val_gold[smoke_qid])
    print(f'  {("[GOLD]" if in_gold else "[----]")}  yes={s:.3f}  {c}')


## Cell 7 — Judge prompt + soft-yes scoring

Prompt structure:
```
<system>: You are a Swiss legal expert ...
<user>:   QUERY (English):        ...
          CANDIDATE CITATION:     Art. 221 Abs. 1 StPO
          STATUTE (German):       ...
          STATUTE TITLE:          ...
          Question: would a Swiss Federal Court drafting the opinion for this query cite this provision?
          Answer YES or NO.
```

We request 1 token + `logprobs=20`, then compute `softmax([logprob_NO, logprob_YES])[1]` as the soft yes-score. This is the same scoring trick the user already uses in `Qwen3-Reranker-8B`.


In [ ]:
# Install vLLM if needed (Colab)
import subprocess, sys
def _have(mod):
    try: __import__(mod); return True
    except ImportError: return False
if not _have('vllm'):
    print('Installing vllm ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm', 'autoawq'], check=False)

JUDGE_MODEL_ID = 'Qwen/Qwen3-14B-AWQ'
FALLBACK_MODEL_ID = 'Qwen/Qwen3-8B-AWQ'

# Try to load vLLM with Qwen3-14B-AWQ; fall back if VRAM tight
LLM, TOK, MODEL_ID = None, None, None
try:
    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer
    try:
        LLM = LLM(model=JUDGE_MODEL_ID, dtype='float16', gpu_memory_utilization=0.85, max_model_len=4096)
        MODEL_ID = JUDGE_MODEL_ID
    except Exception as e:
        print(f'Qwen3-14B-AWQ load failed ({e}); trying Qwen3-8B-AWQ...')
        LLM = LLM(model=FALLBACK_MODEL_ID, dtype='float16', gpu_memory_utilization=0.85, max_model_len=4096)
        MODEL_ID = FALLBACK_MODEL_ID
    TOK = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    print(f'Judge model loaded: {MODEL_ID}')
except Exception as e:
    print(f'vLLM unavailable: {e}')
    print('Continue execution from Cell 7 — judge will skip and use heuristic fallback.')


## Cell 6 — Sophisticated judge: prompt + vLLM (Qwen3-14B-AWQ)

Architecture:
- **Reader-style prompt** in English: query + canonical citation + actual German statute text + title.
- **Per-candidate yes/no with logprobs** → soft confidence in [0,1].
- vLLM batched inference (we send ~30-100 candidates per val query in one batch).
- Caches verdicts to disk so re-running threshold sweeps is free.

If vLLM/Qwen-14B is unavailable, the cell falls back to a `transformers` pipeline with `Qwen3-8B-AWQ` (smaller, fits T4).


In [ ]:
_AC_RE  = re.compile(r'^Art\.\s+(\d+[a-z]?)\b.*?\s+([A-Za-z][A-Za-z]{1,9}\d?)\s*$')
_ABS_RE = re.compile(r'\bAbs\.\s+\d+[a-z]?\b')

def _strip_lit(cit):
    return re.sub(r'\s+lit\.\s+[a-z]\b', '', cit)

def _strip_to_artcode(cit):
    m = _AC_RE.match(cit)
    return f'Art. {m.group(1)} {m.group(2)}' if m else None

def lookup_law_text(cit, max_chars=900):
    """Return (text, title, resolved_citation) — text trimmed to max_chars."""
    # Tier 1: exact
    if cit in LAW_BY_CITATION:
        t, ti = LAW_BY_CITATION[cit]
        return t[:max_chars], ti, cit
    # Tier 2: drop "lit. X"
    c2 = _strip_lit(cit)
    if c2 != cit and c2 in LAW_BY_CITATION:
        t, ti = LAW_BY_CITATION[c2]
        return t[:max_chars], ti, c2
    # Tier 3: same art+code, any paragraph (pick first available)
    k = _strip_to_artcode(cit)
    if k and k in LAW_BY_ARTCODE:
        bucket = LAW_BY_ARTCODE[k]
        # Prefer the entry that matches the predicted Abs. number if any
        m_abs = re.search(r'Abs\.\s+(\d+[a-z]?)', cit)
        if m_abs:
            tgt = m_abs.group(1)
            for cit2, t, ti in bucket:
                if re.search(rf'Abs\.\s+{re.escape(tgt)}\b', cit2):
                    return t[:max_chars], ti, cit2
        # Otherwise concatenate up to 3 paragraphs (so judge sees the article in context)
        chunks = []
        used = []
        for cit2, t, ti in bucket[:3]:
            chunks.append(t)
            used.append(cit2)
        joined = '\n---\n'.join(chunks)
        return joined[:max_chars], bucket[0][2], '|'.join(used)
    return '', '', None

# Sanity check coverage on val candidates
miss = 0
total = 0
for q, cands in law_preds_val.items():
    for c in cands:
        total += 1
        txt, ti, resolved = lookup_law_text(c)
        if not txt:
            miss += 1
print(f'Law-text lookup coverage on val candidates: {total-miss}/{total} ({100*(total-miss)/max(1,total):.1f}%)')

# Show a few examples
print('\nExamples:')
for q in list(law_preds_val)[:1]:
    for c in law_preds_val[q][:4]:
        t, ti, resolved = lookup_law_text(c, max_chars=300)
        print(f'  {c}  -> resolved={resolved!r}, title={ti[:60]!r}')
        print(f'    text: {t[:200]!r}')


## Cell 5 — Law-text lookup (resolve `Art. X Abs. Y CODE` → German text)

For each candidate citation, we need the actual statute text to give the judge. Lookup tiers:
1. Exact citation match in `laws_de.csv`.
2. Same article+code, any-paragraph (e.g. predicted `Art. 100 BGG` → use `Art. 100 Abs. 1 BGG` text).
3. Sibling-paragraph match (e.g. predicted `Art. 221 Abs. 2 lit. b StPO`, `lit. b` not in corpus → fall back to `Art. 221 Abs. 2 StPO`).

If still no match, we keep the candidate without text but pass the citation string to the judge — it can still reason about which legal area the citation points to.


In [ ]:
def _is_law_cite(c):
    return bool(re.match(r'^Art\.\s*\d', c))

# Baseline 1: current law preds with official canonicalization (this is the Kaggle-equivalent number)
print('### Baseline: current law preds (canonical scoring)')
macro_baseline, rows_baseline = official_score(law_preds_val, val_gold, label='LAW-ONLY baseline (canonical)')

# Ceiling: predict exactly the gold law cites, no court
ceiling_preds = {q: [c for c in val_gold[q] if _is_law_cite(c)] for q in val_gold}
print('\n### Ceiling: perfect law cites, zero court')
macro_ceiling, rows_ceiling = official_score(ceiling_preds, val_gold, label='CEILING (perfect law, no court)')

print('\n--- summary ---')
print(f"  current law preds:           macro F1 = {macro_baseline['macro_f1']:.4f}")
print(f"  perfect-law ceiling:         macro F1 = {macro_ceiling['macro_f1']:.4f}")
print(f"  headroom from removing FPs: ~{macro_ceiling['macro_f1'] - macro_baseline['macro_f1']:+.4f}")


## Cell 4 — Baseline scores (official metric, no judge)

Three scenarios:
1. **Current law preds, raw** (no canonicalization)
2. **Current law preds, canonical** (= what Kaggle measures)
3. **Ceiling: perfect law-only** (every gold-law cite, nothing else)


In [ ]:
from omnilex.evaluation.metrics import citation_f1, macro_f1, micro_f1
from omnilex.citations.normalizer import CitationNormalizer

NORMALIZER = CitationNormalizer()

def canon_list(cits):
    """Apply Omnilex canonicalization. Returns a deduped list of canonical IDs.
    Unparseable cites (e.g. '1B_210/2023 E. 4.1') are dropped — matching the official scorer."""
    return NORMALIZER.canonicalize_list(list(cits))

def official_score(pred_per_qid, gold_per_qid, label='', show_per_query=True):
    """Score a {qid: [cites]} prediction dict against {qid: [gold cites]}.
    Returns (macro_scores_dict, per_query_rows)."""
    qids = sorted(gold_per_qid)
    preds_canon = [canon_list(pred_per_qid.get(q, [])) for q in qids]
    golds_canon = [canon_list(gold_per_qid[q]) for q in qids]
    macro = macro_f1(preds_canon, golds_canon)
    micro = micro_f1(preds_canon, golds_canon)
    rows = []
    for q, p, g in zip(qids, preds_canon, golds_canon):
        s = citation_f1(p, g)
        rows.append((q, len(g), len(p), len(set(p)&set(g)), s['precision'], s['recall'], s['f1']))
    if show_per_query:
        print(f'\n=== {label} ===')
        print(f"{'qid':<10}{'|G|':>5}{'|P|':>5}{'TP':>5}{'P':>8}{'R':>8}{'F1':>8}")
        for r in rows:
            print(f"{r[0]:<10}{r[1]:>5}{r[2]:>5}{r[3]:>5}{r[4]:>8.3f}{r[5]:>8.3f}{r[6]:>8.3f}")
        print(f"\n  MACRO  precision={macro['macro_precision']:.4f}  recall={macro['macro_recall']:.4f}  F1={macro['macro_f1']:.4f}")
        print(f"  MICRO  precision={micro['micro_precision']:.4f}  recall={micro['micro_recall']:.4f}  F1={micro['micro_f1']:.4f}")
    return macro, rows

# Sanity check: how much does canonicalization shrink val gold?
total_raw = sum(len(g) for g in val_gold.values())
total_canon = sum(len(canon_list(g)) for g in val_gold.values())
print(f'Val gold: raw={total_raw}, after canonicalization={total_canon}, dropped={total_raw-total_canon} ({100*(total_raw-total_canon)/total_raw:.1f}%)')
print('(Dropped items are unparseable case numbers like 1B_xxx, 7B_xxx that the Omnilex normalizer cannot handle.)')


## Cell 3 — Official Omnilex scoring (canonical macro F1)

This is the **competition metric** as defined in `omnilex.evaluation.metrics`. We import the same module the test grader uses, apply `CitationNormalizer.canonicalize_list` to both predictions and gold, then compute macro F1 = mean per-query F1.


In [ ]:
val  = pd.read_csv(DATA_DIR / 'val.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

def parse_cites(s):
    return [c.strip() for c in str(s).split(';') if c.strip()] if pd.notna(s) else []

val['gold_list'] = val['gold_citations'].apply(parse_cites)
val_query = {r['query_id']: r['query'] for _, r in val.iterrows()}
val_gold  = {r['query_id']: r['gold_list'] for _, r in val.iterrows()}
test_query = {r['query_id']: r['query'] for _, r in test.iterrows()}

# Current law-pipeline predictions (saved by the law notebook)
LAW_PREDS_VAL  = LAW_PIPE_DIR / 'retrieval/pipeline_output_v12/law_predictions_per_query.json'
LAW_PREDS_TEST = LAW_PIPE_DIR / 'retrieval/pipeline_output_v12/law_predictions_per_query_TEST.json'

law_preds_val  = json.loads(LAW_PREDS_VAL.read_text(encoding='utf-8'))  if LAW_PREDS_VAL.exists()  else {}
law_preds_test = json.loads(LAW_PREDS_TEST.read_text(encoding='utf-8')) if LAW_PREDS_TEST.exists() else {}
print(f'Law preds (val): {len(law_preds_val)} | (test): {len(law_preds_test)}')

# Statute text lookup
laws = pd.read_csv(DATA_DIR / 'laws_de.csv', usecols=['citation','text','title'])
laws['text'] = laws['text'].fillna('').astype(str)
laws['title'] = laws['title'].fillna('').astype(str)

# Build two indices:
#  (1) exact citation -> (text, title)
#  (2) article+code (canonical no-Abs) -> list of (citation, text, title)
LAW_BY_CITATION = {}
LAW_BY_ARTCODE  = defaultdict(list)
def _artcode_key(cit):
    # Strip optional 'Abs. ...' / 'lit. ...' from "Art. X Abs. Y lit. z CODE"
    m = re.match(r'^Art\.\s+(\d+[a-z]?)\b.*?\s+([A-Za-z][A-Za-z]{1,9}\d?)\s*$', cit)
    if not m: return None
    return f'Art. {m.group(1)} {m.group(2)}'

for _, r in laws.iterrows():
    cit = r['citation']
    LAW_BY_CITATION[cit] = (r['text'], r['title'])
    k = _artcode_key(cit)
    if k:
        LAW_BY_ARTCODE[k].append((cit, r['text'], r['title']))

print(f'laws_de.csv: {len(laws)} rows, {len(LAW_BY_CITATION)} unique citations, {len(LAW_BY_ARTCODE)} unique art+code keys')

# Per-query: how many law candidates does the pipeline currently emit?
print('\nLaw-pipeline candidate counts per val query:')
for qid in sorted(val_query):
    cands = law_preds_val.get(qid, [])
    n_in_gold = sum(1 for c in cands if c in set(val_gold[qid]))
    print(f'  {qid}: {len(cands):3d} predicted | {n_in_gold:3d} match gold (raw)')


## Cell 2 — Load val, law predictions, laws_de.csv (statute text lookup)


In [ ]:
import os, re, json, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd, numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive')
    IN_COLAB = True
except Exception:
    DRIVE = Path('E:/swiss_citation_extraction/drive_sync')
    IN_COLAB = False
    print('Not Colab — using local drive_sync mirror')

SWISS_LAW_DIR    = DRIVE / 'swiss_law'
LAW_PIPE_DIR     = DRIVE / 'Omnilex-Agentic-Retrieval-Competition'
OMNILEX_REPO_DIR = DRIVE / 'Omnilex-Agentic-Retrieval-Competition'  # the cloned repo, with src/omnilex/...
DATA_DIR         = SWISS_LAW_DIR / 'data'
OUT_DIR          = SWISS_LAW_DIR / 'law_judge_2026-05-23'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If the Omnilex source tree is at a different path on your Drive, fix it here.
# Search common locations:
for cand in [
    OMNILEX_REPO_DIR / 'src',
    DRIVE / 'Omnilex-Agentic-Retrieval-Competition' / 'src',
    DRIVE / 'swiss_law' / 'Omnilex-Agentic-Retrieval-Competition' / 'src',
]:
    if (cand / 'omnilex' / 'evaluation' / 'metrics.py').exists():
        sys.path.insert(0, str(cand))
        print(f'Omnilex on path: {cand}')
        break
else:
    print('WARNING: Omnilex repo not found on Drive — clone it next to the data folder.')

print('DATA_DIR:', DATA_DIR)
print('OUT_DIR :', OUT_DIR)


## Cell 1 — Setup (paths, Drive mount, dependencies)


# Law judge with official Omnilex scoring — 2026-05-23

**Purpose.** Filter the current law-pipeline predictions on val using a sophisticated LLM judge that reads each candidate's German statute text alongside the English query, then score with the *official* Omnilex normalizer + macro F1.

**Pipeline.**

1. Load current law predictions per val query (output of the existing Qwen3-Reranker → Qwen3-8B judge pipeline).
2. For each (query, predicted-law-citation) pair, look up the German statute text from `laws_de.csv` and ask Qwen3-14B-AWQ (via vLLM): *"Is this provision necessary to answer this Swiss legal query?"*
3. Keep only YES verdicts, retaining the judge's confidence (from the YES logprob).
4. Score the filtered predictions with the official `omnilex.evaluation.macro_f1`, after applying `CitationNormalizer.canonicalize_list` to both predictions and gold.

**Why this should help.** The current law-only pipeline scores LAW-F1 = 0.777 on val but joint F1 = 0.585. The gap is dominated by **false positives** in the law set (predictions that match no gold citation). A second-pass judge that reads the actual statute text — not just the citation string — should remove provisions whose text doesn't address the query's facts.

**Ceiling.** Perfect law-only prediction (every gold law cite, zero court) caps at macro F1 = 0.772 on raw gold, ~0.82 on canonicalized gold (the official scorer drops ~13% of gold — Federal Tribunal case numbers like `1B_210/2023` that the Omnilex normalizer can't parse).
